# Combine csv files 

In [1]:
# combine excel sheets from scraping
import os
import glob
import pandas as pd

In [8]:
# path = ' #need full file path '
path = 'D:\Programming\Jupyter\IS453 - Financial Analytics\Github\Twitter'
extension = 'csv'
os.chdir(path)
output_filename = path.split('/')[-1] + '_combined.csv'

all_files = [i for i in glob.glob('*.{}'.format(extension))]

combined_csv = pd.concat([pd.read_csv(f) for f in all_files])
combined_csv.to_csv(output_filename, index = False, encoding = 'utf-8-sig')

# Cleaning posts

In [3]:
import re
import nltk
import numpy as np
import pandas as pd
from nltk import word_tokenize
from nltk.tokenize import RegexpTokenizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

In [4]:
# text cleaning
def text_lower(text):
    return text.lower()

def remove_hashtag_mentions_urls(text):
    return re.sub(r"(?:\@|\#|https?\://)\S+", "", text)

def remove_emoji(text):
    emoji_pattern = re.compile("["
    u"\U0001F600-\U0001F64F"  # emoticons
    u"\U0001F300-\U0001F5FF"  # symbols & pictographs
    u"\U0001F680-\U0001F6FF"  # transport & map symbols
    u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
    u"\U00002702-\U000027B0"
    u"\U000024C2-\U0001F251"
    "]+", flags=re.UNICODE)

    return emoji_pattern.sub(r'', text)

def tokenization(text):
    word_tokenizer = RegexpTokenizer(r'[-\'\w]+')
    tokenized_text = word_tokenizer.tokenize(text)
    return tokenized_text

# News

In [5]:
def clean_news(filepath):
    df = pd.read_csv(path)

    #cleaning summary (mostly just tokenization)
    summary = df['Summary'].to_list()
    cleaned_summary = []

    for text in summary:
        try:
            text = text_lower(text)
            text = remove_hashtag_mentions_urls(text)
            text = remove_emoji(text)
            text = tokenization(text)
            cleaned_summary.append(' '.join(text))
        except:
            cleaned_summary.append('')

    df['cleaned_summary'] = cleaned_summary


    #cleaning title
    title = df['Title'].to_list()
    cleaned_title = []

    for text in title:
        try:
            text = text_lower(text)
            text = remove_hashtag_mentions_urls(text)
            text = remove_emoji(text)
            text = tokenization(text)
            cleaned_title.append(' '.join(text))
        except:
            cleaned_title.append('')

    df['cleaned_title'] = cleaned_title
    
    #removing missing values from updated df
    df['cleaned_summary'].replace('', np.nan, inplace = True)
    df['cleaned_title'].replace('', np.nan, inplace = True)
    df.dropna(subset=['cleaned_summary', 'cleaned_title'], inplace = True)
    
    df.to_csv(filepath[:-4] + '_cleaned.csv')
    

# Twitter

In [30]:
def clean_twitter(filepath):
    df = pd.read_csv(filepath)
    
    #cleaning tweet
    tweets = df['Tweet'].to_list()
    cleaned_tweets = []
    
    for tweet in tweets:
        try:
            tweet = text_lower(tweet)
            tweet = remove_hashtag_mentions_urls(tweet)
            tweet = remove_emoji(tweet)
            tweet = tokenization(tweet)
            cleaned_tweets.append(' '.join(tweet))
        except:
            cleaned_tweets.append('')
    
    df['cleaned_tweets'] = cleaned_tweets
    
    #removing missing values from updated df
    df['cleaned_tweets'].replace('', np.nan, inplace=True)
    df.dropna(subset=['cleaned_tweets'], inplace = True)
    
    #removing tweets with lesser than 5words
    df = df[df['cleaned_tweets'].apply(lambda x: len(x)>4)]
    df = df[df['Number of Likes'] > 200]
    
    df.to_csv(filepath[:-4] + '_cleaned.csv')


In [31]:
# path = path + '/' + output_filename
path = 'D:/Programming/Jupyter/IS453 - Financial Analytics/Github/Twitter_combined.csv'
clean_twitter(path)

import flair
sentiment_model = flair.models.TextClassifier.load('en-sentiment')